---
---
# Universidad Federico Santa María - 2026

<img src="https://fdiaz1968.github.io/Finance-MBA/images/logo_utfsm.png" alt="Universidad Técnica Federico Santa María - Departamento de Ingeniería Comercial" style="width: 480px !important; max-width: 100% !important; height: auto !important;"/>

## FINANZAS

### Profesor Fernando Díaz H.
---

# 🏭 El Modelo de Tres Factores de Fama-French

El **CAPM** explica el retorno esperado de una acción con un solo factor: su beta de mercado. Sin embargo, desde fines de los años 80 la evidencia empírica mostró que el CAPM deja retornos sistemáticamente sin explicar: las acciones de empresas **pequeñas** y las acciones **"valor"** (con un alto ratio *book-to-market*) tienden a rendir, en promedio, más de lo que su beta de mercado predice. Es decir, sus **alfas de Jensen** son significativamente positivos.

Fama y French (1993) propusieron explicar esos retornos "anómalos" agregando dos factores adicionales al CAPM:

$$R_{i,t}=\alpha_{i}+\beta_{i}\,MRP_{t}+s_{i}\,SMB_{t}+h_{i}\,HML_{t}+\varepsilon_{i,t}$$

donde $R_{i,t}=r_{i,t}-r_{f,t}$ es el retorno **en exceso** de la acción, y:

* $MRP_{t}=r_{M,t}-r_{f,t}$ es la prima de mercado (el mismo factor del CAPM, también llamado `Mkt-RF`);
* $SMB_{t}$ (*Small Minus Big*) es la prima por **tamaño**: el retorno de un portafolio de acciones pequeñas menos uno de acciones grandes;
* $HML_{t}$ (*High Minus Low*) es la prima por **valor**: el retorno de un portafolio de acciones "valor" (alto *book-to-market*) menos uno de acciones "crecimiento" (bajo *book-to-market*);
* $s_{i}$ y $h_{i}$ son las sensibilidades (*loadings*) de la acción a cada factor.

Como la regresión ya está escrita en retornos **en exceso**, el intercepto $\alpha_{i}$ es, igual que en el notebook de estimación de betas, el **alfa de Jensen**: el retorno que el modelo no logra explicar. La diferencia es que ahora "el modelo" no es solo el mercado, sino los tres factores en conjunto.

En este notebook:

1. Descargaremos los factores de Fama-French directamente desde el sitio de **Kenneth French** (Dartmouth).
2. Descargaremos precios de cinco acciones y calcularemos sus retornos mensuales en exceso.
3. Estimaremos, para cada acción, el **modelo de mercado (CAPM)** y el **modelo de tres factores**, ambos con retornos en exceso.
4. Compararemos el alfa de Jensen del CAPM con el alfa del modelo de tres factores, para ver si SMB y HML explican parte de lo que el CAPM dejaba como "anómalo".

> 💡 **Idea central:** si una acción tiene un alfa de Jensen positivo en el CAPM simplemente porque es una acción pequeña o "valor" —no porque el mercado la esté "sub-valorando"—, entonces ese alfa debería **reducirse o desaparecer** al controlar por SMB y HML.

## 📦 Cargando las librerías

* **`tidyquant`**: descarga de precios de acciones.
* **`readr`**, **`lubridate`**: lectura y manejo de fechas del archivo de factores.
* **`tidyr`**, **`dplyr`**: transformación de datos.
* **`ggplot2`**, **`scales`**: gráficos.
* **`broom`**: tablas ordenadas a partir de los resultados de una regresión.
* **`stargazer`**: tablas de regresión con formato académico.

In [ ]:
#install.packages(c("tidyquant", "broom", "stargazer"))

In [ ]:
suppressWarnings(suppressPackageStartupMessages({
  library(tidyquant)
  library(readr)
  library(lubridate)
  library(tidyr)
  library(dplyr)
  library(ggplot2)
  library(scales)
  library(broom)
  library(stargazer)
}))

options(repr.plot.width = 10, repr.plot.height = 6, repr.plot.res = 150)

---
## 📥 Descargando los factores de Fama-French

Los factores se publican en el sitio de **Kenneth French** como un archivo `.zip` que contiene un `.csv`. Lo descargamos a un archivo temporal, lo descomprimimos y lo leemos, saltando las tres primeras filas de encabezado.

El archivo trae primero los datos **mensuales** (fecha como `AAAAMM`, 6 dígitos) y, más abajo, tras un salto, los datos **anuales** (fecha como `AAAA`, 4 dígitos). En vez de fijar a mano el número de filas mensuales —que cambia cada vez que Dartmouth actualiza el archivo—, nos quedamos únicamente con las filas cuya fecha tiene **6 dígitos**.

Los factores vienen en **porcentaje**, por lo que los dividimos por 100, y renombramos `Mkt-RF` como `MRP`, para ser consistentes con la notación del notebook de estimación de betas.

In [ ]:
ff_url <- "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_CSV.zip"

temp_zip <- tempfile(fileext = ".zip")
download.file(ff_url, temp_zip, quiet = TRUE)
ff_csv <- unzip(temp_zip, exdir = tempdir())

ff_raw <- read_csv(ff_csv, skip = 3, show_col_types = FALSE) |>
  rename(date = 1)

ff_factors <- ff_raw |>
  filter(!is.na(date), nchar(trimws(as.character(date))) == 6) |>
  mutate(
    date = ymd(paste0(date, "01")),
    MRP  = `Mkt-RF` / 100,
    SMB  = SMB / 100,
    HML  = HML / 100,
    RF   = RF / 100
  ) |>
  filter(date >= as.Date("2020-12-01"), date <= as.Date("2025-12-01")) |>
  select(date, MRP, SMB, HML, RF)

cat(sprintf("Observaciones mensuales: %d (de %s a %s)\n",
            nrow(ff_factors), min(ff_factors$date), max(ff_factors$date)))
head(ff_factors)

## 📥 Descargando los precios de las acciones

Trabajaremos con cinco acciones de sectores distintos:

* **Apple Inc. (`AAPL`)**: tecnología.
* **Walmart Inc. (`WMT`)**: comercio minorista.
* **Intel Corporation (`INTC`)**: semiconductores.
* **Exxon Mobil Corporation (`XOM`)**: energía.
* **Lockheed Martin Corporation (`LMT`)**: aeroespacial y defensa.

Al igual que en el notebook de estimación de betas, partimos en **diciembre de 2020** para disponer del precio base de enero de 2021, y trabajamos con retornos logarítmicos **mensuales**.

In [ ]:
tick <- c("AAPL", "WMT", "INTC", "XOM", "LMT")

price_data <- tq_get(tick,
                     from = "2020-12-01",
                     to   = "2026-01-01",
                     get  = "stock.prices")

log_ret <- price_data |>
  group_by(symbol) |>
  tq_transmute(select = adjusted, mutate_fun = periodReturn,
               period = "monthly", col_rename = "ret", type = "log") |>
  ungroup() |>
  mutate(date = floor_date(date, "month")) |>
  filter(date >= as.Date("2021-01-01")) |>
  pivot_wider(names_from = symbol, values_from = ret)

head(log_ret)

---
## 🔗 Combinando retornos y factores

Unimos los retornos de las acciones con los factores de Fama-French por fecha, y calculamos el retorno **en exceso** de cada acción restando `RF` (la tasa libre de riesgo mensual que reporta el propio Kenneth French).

> 💡 A diferencia del notebook de estimación de betas, aquí **no** necesitamos calcular nosotros mismos la tasa libre de riesgo ni el retorno del mercado: ambos vienen incluidos en el archivo de factores (`RF` y `MRP`, respectivamente), construidos y validados por Kenneth French a partir de todo el universo de acciones de EE.UU. `SMB` y `HML` se construyen ordenando, cada junio, todas las acciones de EE.UU. por tamaño y por *book-to-market*, formando 6 portafolios de referencia (small/big x low/medium/high) con los puntos de corte de NYSE; `SMB` es el retorno promedio de los portafolios pequeños menos los grandes, y `HML` el de los portafolios "valor" menos los "crecimiento". No replicamos ese procedimiento aquí: lo usamos ya construido.

In [ ]:
db <- inner_join(log_ret, ff_factors, by = "date") |>
  mutate(across(all_of(tick), ~ .x - RF))

cat(sprintf("Observaciones combinadas: %d\n", nrow(db)))
head(db)

## 📊 Estadística descriptiva de los factores

Antes de estimar, veamos el comportamiento de los tres factores durante la muestra, con `stargazer`.

In [ ]:
stargazer(as.data.frame(select(db, MRP, SMB, HML)), type = "text",
          title = "Estadística Descriptiva de los Factores", digits = 4)

### 📈 Evolución acumulada de los factores

Grafiquemos el crecimiento de $1 invertido en cada factor a lo largo de la muestra ($e^{\sum r_{t}}$, ya que trabajamos con retornos logarítmicos). Es habitual que `MRP` muestre una tendencia positiva y volátil, mientras que `SMB` y `HML` fluctúan mucho más cerca de cero.

> ⚠️ Un `SMB` o `HML` con tendencia negativa en un período dado no invalida el modelo: son primas de riesgo **esperadas** en promedio a largo plazo, no retornos garantizados en cualquier ventana de cinco años.

In [ ]:
db |>
  select(date, MRP, SMB, HML) |>
  mutate(across(c(MRP, SMB, HML), ~ exp(cumsum(.x)))) |>
  pivot_longer(-date, names_to = "Factor", values_to = "Valor") |>
  ggplot(aes(x = date, y = Valor, color = Factor)) +
  geom_hline(yintercept = 1, linetype = "dotted", color = "gray40") +
  geom_line(linewidth = 1) +
  scale_y_continuous(labels = dollar_format(prefix = "$")) +
  labs(x = NULL, y = "Crecimiento de $1 invertido",
       title = "Evolución acumulada de los factores de Fama-French") +
  theme_minimal()

---
## 🧮 Modelo de Mercado (CAPM) con retornos en exceso

Primero, para cada acción, estimamos el modelo de mercado clásico —solo `MRP` como regresor—, igual que en el notebook de estimación de betas. Como ya trabajamos con retornos en exceso, el intercepto $\hat{\alpha}$ es el **alfa de Jensen**.

In [ ]:
modelo_capm_aapl <- lm(AAPL ~ MRP, data = db)
modelo_capm_wmt  <- lm(WMT  ~ MRP, data = db)
modelo_capm_intc <- lm(INTC ~ MRP, data = db)
modelo_capm_xom  <- lm(XOM  ~ MRP, data = db)
modelo_capm_lmt  <- lm(LMT  ~ MRP, data = db)

modelos_capm <- list(AAPL = modelo_capm_aapl, WMT = modelo_capm_wmt, INTC = modelo_capm_intc,
                     XOM = modelo_capm_xom, LMT = modelo_capm_lmt)

periodo <- sprintf("%s a %s", format(min(db$date), "%Y-%m"), format(max(db$date), "%Y-%m"))

stargazer(modelos_capm, type = "text",
          title = paste("Modelo de Mercado (CAPM),", periodo), align = TRUE)

In [ ]:
tabla_capm <- bind_rows(lapply(names(modelos_capm), function(a) {
  m  <- modelos_capm[[a]]
  al <- filter(tidy(m), term == "(Intercept)")
  be <- filter(tidy(m), term == "MRP")
  tibble(Accion = a, Alfa_mensual = al$estimate, Alfa_anual = al$estimate * 12,
         Valor_p = al$p.value, Beta_MRP = be$estimate, R2 = glance(m)$r.squared)
}))

tabla_capm |> mutate(across(where(is.numeric), ~ round(.x, 4)))

---
## 🧮 Modelo de Tres Factores (Fama-French)

Ahora agregamos `SMB` y `HML` como regresores adicionales. El intercepto de esta regresión —que llamaremos $\alpha^{FF3}$— es también un alfa de Jensen, pero **ajustado por tamaño y valor**: mide el retorno que ni el mercado, ni el tamaño, ni el valor de la acción logran explicar.

In [ ]:
modelo_ff3_aapl <- lm(AAPL ~ MRP + SMB + HML, data = db)
modelo_ff3_wmt  <- lm(WMT  ~ MRP + SMB + HML, data = db)
modelo_ff3_intc <- lm(INTC ~ MRP + SMB + HML, data = db)
modelo_ff3_xom  <- lm(XOM  ~ MRP + SMB + HML, data = db)
modelo_ff3_lmt  <- lm(LMT  ~ MRP + SMB + HML, data = db)

modelos_ff3 <- list(AAPL = modelo_ff3_aapl, WMT = modelo_ff3_wmt, INTC = modelo_ff3_intc,
                    XOM = modelo_ff3_xom, LMT = modelo_ff3_lmt)

stargazer(modelos_ff3, type = "text",
          title = paste("Modelo de Tres Factores (Fama-French),", periodo), align = TRUE)

In [ ]:
tabla_ff3 <- bind_rows(lapply(names(modelos_ff3), function(a) {
  m  <- modelos_ff3[[a]]
  td <- tidy(m)
  al <- filter(td, term == "(Intercept)")
  tibble(Accion = a, Alfa_FF3_mensual = al$estimate, Alfa_FF3_anual = al$estimate * 12,
         Valor_p = al$p.value,
         Beta_MRP = filter(td, term == "MRP")$estimate,
         Beta_SMB = filter(td, term == "SMB")$estimate,
         Beta_HML = filter(td, term == "HML")$estimate,
         R2 = glance(m)$r.squared)
}))

tabla_ff3 |> mutate(across(where(is.numeric), ~ round(.x, 4)))

### 📖 ¿Cómo leer los *loadings* $s_{i}$ y $h_{i}$?

* $s_{i}>0$: la acción se mueve como una acción **pequeña** (junto con `SMB`); $s_{i}<0$, como una acción **grande**.
* $h_{i}>0$: la acción se comporta como una acción **"valor"** (junto con `HML`); $h_{i}<0$, como una acción **"crecimiento"** —empresas con altas expectativas de crecimiento futuro y un *book-to-market* bajo, típicamente tecnológicas.

> ⚠️ Con solo 60 observaciones y tres regresores correlacionados entre sí, los errores estándar de estos *loadings* suelen ser amplios. Interprete el **signo** con más confianza que la magnitud exacta.

### 📄 Exportando las tablas a LaTeX (para las diapositivas)

Las tablas de esta seccion alimentan directamente las diapositivas del curso ("Regresiones de F&F", "Regresiones Market Model"). Como el periodo de la muestra puede cambiar de un semestre a otro, generamos el codigo LaTeX directamente desde los datos actuales, en vez de copiarlo a mano: asi el titulo y el numero de observaciones de la tabla siempre reflejan la muestra que efectivamente se uso.

> 💡 Copie el contenido de `Market_Model.tex` y `Fama_French.tex` directamente dentro de los bloques `\\begin{table}...\\end{table}` de la presentacion, reemplazando la tabla anterior.

In [ ]:
stargazer(modelos_capm, type = "latex", title = paste("Modelo de Mercado,", periodo),
          align = TRUE, out = "Market_Model.tex")

stargazer(modelos_ff3, type = "latex", title = paste("Modelo de Tres Factores (Fama-French),", periodo),
          align = TRUE, out = "Fama_French.tex")

cat("Tablas exportadas: Market_Model.tex, Fama_French.tex\n")

---
## 🔍 Comparando el alfa del CAPM con el alfa del modelo de tres factores

Si una acción tenía un alfa de Jensen positivo en el CAPM **porque** es una acción pequeña o "valor" —y no porque el mercado la esté sub-valorando—, entonces su alfa debería **reducirse** al pasar al modelo de tres factores, que ya controla por esas dos características.

In [ ]:
comparacion_alfas <- tibble(
  Accion = tabla_capm$Accion,
  Alfa_CAPM_anual = tabla_capm$Alfa_anual,
  Alfa_FF3_anual  = tabla_ff3$Alfa_FF3_anual,
  R2_CAPM = tabla_capm$R2,
  R2_FF3  = tabla_ff3$R2
) |>
  mutate(Reduccion_alfa = Alfa_CAPM_anual - Alfa_FF3_anual)

comparacion_alfas |> mutate(across(where(is.numeric), ~ round(.x, 4)))

In [ ]:
comparacion_alfas |>
  pivot_longer(c(Alfa_CAPM_anual, Alfa_FF3_anual), names_to = "Modelo", values_to = "Alfa") |>
  mutate(Modelo = recode(Modelo, Alfa_CAPM_anual = "Alfa CAPM", Alfa_FF3_anual = "Alfa Fama-French (3F)")) |>
  ggplot(aes(x = Accion, y = Alfa, fill = Modelo)) +
  geom_col(position = position_dodge(width = 0.7), width = 0.6, color = "black") +
  geom_hline(yintercept = 0) +
  scale_fill_manual(values = c("Alfa CAPM" = "steelblue", "Alfa Fama-French (3F)" = "#d62728")) +
  scale_y_continuous(labels = percent_format(accuracy = 1)) +
  labs(x = NULL, y = "Alfa anualizado", title = "Alfa de Jensen: CAPM vs. Modelo de Tres Factores") +
  theme_minimal() +
  theme(legend.position = "bottom", legend.title = element_blank())

> 💡 **Cómo leer este gráfico:** si la barra roja (FF3) es más chica que la azul (CAPM) y ambas tienen el mismo signo, parte del alfa "CAPM" se explica por la exposición de la acción a `SMB` y `HML`. Si el alfa FF3 sigue siendo grande —y estadísticamente distinto de cero—, el modelo de tres factores tampoco logra explicar el retorno de esa acción.

---
## ⚠️ Precauciones metodológicas

* **El factor de mercado no es el mismo que en el CAPM del notebook anterior.** Aquí `MRP` es el `Mkt-RF` que publica Kenneth French —construido con (casi) todas las acciones que cotizan en EE.UU.—, mientras que en el notebook de estimación de betas usamos el S&P 500 como proxy. Ambos son razonables, pero no son idénticos.
* **Multicolinealidad.** `MRP`, `SMB` y `HML` no son independientes entre sí, lo que puede inflar los errores estándar de los *loadings* individuales, aun cuando el modelo en su conjunto ajuste bien.
* **Muestra pequeña.** Con 60 observaciones mensuales y tres regresores, hay pocos grados de libertad; los alfas y *loadings* estimados son ruidosos.
* **El modelo de tres factores tampoco es "la verdad".** Es una mejora empírica sobre el CAPM, no una teoría de equilibrio derivada de primeros principios como el CAPM. Existen extensiones con más factores (momentum, calidad, rentabilidad, inversión) que explican aún mejor los retornos.

### 🧭 Conclusión

Al escribir el modelo en retornos **en exceso**, el intercepto de cualquiera de estas regresiones —CAPM o Fama-French— es un alfa de Jensen: el retorno que el modelo, con los factores que incluye, no logra explicar.

El modelo de tres factores agrega `SMB` y `HML` al CAPM. Si el alfa de una acción se reduce al pasar del CAPM al modelo de tres factores, parte de lo que parecía un retorno "anormal" era, en realidad, una prima por tamaño o por valor que el CAPM, con un solo factor, no podía capturar.